# Poisonous Loan Approvals
This notebook investigates how vulnerable a Decision Tree loan approval model is to data poisoning attacks, and whether Isolation Forest can detect and remove the poisoned samples to restore performance.

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import eli5
from lime.lime_tabular import LimeTabularExplainer

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
)

## Load Dataset
We use a loan application dataset with 45,000 records and 14 features.

In [ ]:
df = pd.read_csv("loan_data.csv")
print(f"Shape: {df.shape}")
df.head()

## Exploratory Data Analysis
### Data Types and Basic Statistics

In [ ]:
print("Data types:")
print(df.dtypes)
# print()
df.describe()

### Categorical Features

In [ ]:
cat_cols = df.select_dtypes(include="object").columns.tolist()
print(f"Categorical columns: {cat_cols}\n")
for col in cat_cols:
    print(f"{col}: {df[col].unique()}")
    print(f"  value counts:\n{df[col].value_counts()}\n")

### Feature Distributions

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.ravel()
for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=30, edgecolor="black", alpha=0.7)
    axes[i].set_title(col)
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle("Distributions of Numerical Features", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

### Missing Values and Class Balance

In [ ]:
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "No missing values found")
print(f"\nTotal missing cells: {missing.sum()}")
print(f"Total cells: {df.size}")
print(f"Missing %: {missing.sum() / df.size * 100:.2f}%")

print("\nClass distribution (loan_status):")
counts = df["loan_status"].value_counts()
print(counts)
print(f"\nImbalance ratio (majority / minority): {counts.max() / counts.min():.2f}")

fig, ax = plt.subplots(figsize=(5, 4))
counts.plot.bar(ax=ax, color=["steelblue", "salmon"], edgecolor="black")
ax.set_title("Class Distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Count")
for i, v in enumerate(counts):
    ax.text(i, v + 200, str(v), ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

### Outlier Cleaning
The dataset contains unrealistic values like applicants aged 120+. We remove these before training.

In [ ]:
print(f"Before cleaning: {df.shape[0]} rows")

# remove unrealistic ages (e.g. 100+)
df = df[df["person_age"] <= 100]

# remove extreme employment experience outliers
df = df[df["person_emp_exp"] <= 60]

print(f"After cleaning: {df.shape[0]} rows")
print(f"Removed {45000 - df.shape[0]} outlier rows")

### Encode Categorical Variables
We use LabelEncoder to convert categorical columns to numeric values.

In [ ]:
label_encoders = {}
df_enc = df.copy()

for col in cat_cols:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col])
    label_encoders[col] = le
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

print(f"\nEncoded shape: {df_enc.shape}")
df_enc.head()

### Train/Test Split
We use an 80/20 stratified split to preserve the class distribution.

In [ ]:
TARGET = "loan_status"
X = df_enc.drop(columns=[TARGET])
y = df_enc[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")
print(f"Train class dist:\n{y_train.value_counts(normalize=True).round(4)}")
print(f"\nTest class dist:\n{y_test.value_counts(normalize=True).round(4)}")

## Baseline Decision Tree
### Cross-Validation
We evaluate the model with stratified 5-fold cross-validation.

In [ ]:
dt = DecisionTreeClassifier(random_state=42, class_weight="balanced")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = ["f1", "precision", "recall", "accuracy"]
cv_results = cross_validate(dt, X_train, y_train, cv=skf, scoring=scoring,
                            return_train_score=False)

print("Stratified 5-Fold CV Results:\n")
for metric in scoring:
    scores = cv_results[f"test_{metric}"]
    print(f"{metric:>10s}:  mean={scores.mean():.4f}  std={scores.std():.4f}")

### Test Set Evaluation

In [ ]:
dt.fit(X_train, y_train)
y_pred = dt.predict(X_test)

test_acc  = accuracy_score(y_test, y_pred)
test_f1   = f1_score(y_test, y_pred)
test_prec = precision_score(y_test, y_pred)
test_rec  = recall_score(y_test, y_pred)

# specificity = TN / (TN + FP)
cm_baseline = confusion_matrix(y_test, y_pred)
test_spec = cm_baseline[0, 0] / (cm_baseline[0, 0] + cm_baseline[0, 1])

print("Test Set Results:\n")
print(f"  Accuracy   : {test_acc:.4f}")
print(f"  F1         : {test_f1:.4f}")
print(f"  Precision  : {test_prec:.4f}")
print(f"  Recall     : {test_rec:.4f}")
print(f"  Specificity: {test_spec:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=['Rejected (0)', 'Approved (1)']))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["Rejected", "Approved"], cmap="Blues", ax=ax)
ax.set_title("Confusion Matrix - Baseline")
plt.tight_layout()
plt.show()

### Feature Importance

In [ ]:
importances = pd.Series(dt.feature_importances_, index=X.columns).sort_values(ascending=False)

print(f"Tree depth: {dt.get_depth()}")
print(f"Number of leaves: {dt.get_n_leaves()}\n")
print(importances.round(4).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
importances.plot.barh(ax=ax, color="steelblue", edgecolor="black")
ax.set_title("Feature Importance - Decision Tree")
ax.set_xlabel("Importance")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Data Poisoning Attacks
We implement two attack strategies to test how robust the baseline model is:
- **Label Flipping** — randomly flip a percentage of training labels (0→1 or 1→0)
- **Data Injection** — generate synthetic samples that look like rejected loans but label them as approved

### Label Flipping Attack
We flip 5%, 10%, and 20% of training labels at random and retrain the model each time.

In [ ]:
def label_flipping_attack(X_tr, y_tr, flip_pct, seed=42):
    """Flip a percentage of training labels at random."""
    rng = np.random.RandomState(seed)
    y_poisoned = y_tr.copy()
    n_flip = int(len(y_tr) * flip_pct)
    flip_idx = rng.choice(y_tr.index, size=n_flip, replace=False)
    y_poisoned.loc[flip_idx] = 1 - y_poisoned.loc[flip_idx]
    return y_poisoned, flip_idx

flip_rates = [0.05, 0.10, 0.20]
flip_results = {}

for rate in flip_rates:
    y_poisoned, flipped = label_flipping_attack(X_train, y_train, rate)
    orig_counts = y_train.loc[flipped].value_counts().to_dict()

    dt_poison = DecisionTreeClassifier(random_state=42, class_weight="balanced")
    dt_poison.fit(X_train, y_poisoned)
    y_pred_p = dt_poison.predict(X_test)

    acc  = accuracy_score(y_test, y_pred_p)
    f1   = f1_score(y_test, y_pred_p)
    prec = precision_score(y_test, y_pred_p)
    rec  = recall_score(y_test, y_pred_p)

    flip_results[rate] = {
        "n_flipped": len(flipped),
        "accuracy": round(acc, 4), "f1": round(f1, 4),
        "precision": round(prec, 4), "recall": round(rec, 4),
        "cm": confusion_matrix(y_test, y_pred_p),
    }

    print(f"Flip rate: {rate*100:.0f}%  ({len(flipped)} labels flipped)")
    print(f"  Accuracy : {acc:.4f}  (baseline {test_acc:.4f}, delta={acc - test_acc:+.4f})")
    print(f"  F1       : {f1:.4f}  (baseline {test_f1:.4f}, delta={f1 - test_f1:+.4f})")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall   : {rec:.4f}\n")

### Label Flipping - Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, len(flip_rates) + 1, figsize=(5 * (len(flip_rates) + 1), 4))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred,
    display_labels=["Rejected", "Approved"], cmap="Blues", ax=axes[0])
axes[0].set_title(f"Baseline\nAcc={test_acc:.3f}  F1={test_f1:.3f}")

for i, rate in enumerate(flip_rates):
    cm = flip_results[rate]["cm"]
    ConfusionMatrixDisplay(cm, display_labels=["Rejected", "Approved"]).plot(
        cmap="Reds", ax=axes[i + 1])
    axes[i + 1].set_title(
        f"Flip {rate*100:.0f}%\n"
        f"Acc={flip_results[rate]['accuracy']:.3f}  F1={flip_results[rate]['f1']:.3f}")

plt.suptitle("Label Flipping - Confusion Matrices", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Data Injection Attack
We inject synthetic samples that look like rejected loans but label them as approved, at 5%, 10%, and 20% rates.

In [ ]:
def data_injection_attack(X_tr, y_tr, inject_pct, seed=42):
    """
    Create synthetic samples from class-0 features but label them as class-1.
    This simulates an attacker trying to get risky loans approved.
    """
    rng = np.random.RandomState(seed)
    n_inject = int(len(y_tr) * inject_pct)

    class0_idx = y_tr[y_tr == 0].index
    sampled_idx = rng.choice(class0_idx, size=n_inject, replace=True)
    X_synthetic = X_tr.loc[sampled_idx].copy().reset_index(drop=True)

    # add small noise so they're not exact copies
    num_cols = X_synthetic.select_dtypes(include=np.number).columns
    for col in num_cols:
        noise = rng.normal(0, X_synthetic[col].std() * 0.05, size=n_inject)
        X_synthetic[col] = X_synthetic[col] + noise

    y_synthetic = pd.Series(np.ones(n_inject, dtype=int), name=y_tr.name)

    X_poisoned = pd.concat([X_tr, X_synthetic], ignore_index=True)
    y_poisoned = pd.concat([y_tr, y_synthetic], ignore_index=True)

    return X_poisoned, y_poisoned, n_inject

inject_rates = [0.05, 0.10, 0.20]
inject_results = {}

for rate in inject_rates:
    X_poisoned, y_poisoned, n_injected = data_injection_attack(X_train, y_train, rate)

    dt_inject = DecisionTreeClassifier(random_state=42, class_weight="balanced")
    dt_inject.fit(X_poisoned, y_poisoned)
    y_pred_i = dt_inject.predict(X_test)

    acc  = accuracy_score(y_test, y_pred_i)
    f1   = f1_score(y_test, y_pred_i)
    prec = precision_score(y_test, y_pred_i)
    rec  = recall_score(y_test, y_pred_i)

    inject_results[rate] = {
        "n_injected": n_injected,
        "total_train_size": len(y_poisoned),
        "accuracy": round(acc, 4), "f1": round(f1, 4),
        "precision": round(prec, 4), "recall": round(rec, 4),
        "cm": confusion_matrix(y_test, y_pred_i),
    }

    print(f"Injection rate: {rate*100:.0f}%  ({n_injected} samples, total train={len(y_poisoned)})")
    print(f"  Accuracy : {acc:.4f}  (baseline {test_acc:.4f}, delta={acc - test_acc:+.4f})")
    print(f"  F1       : {f1:.4f}  (baseline {test_f1:.4f}, delta={f1 - test_f1:+.4f})")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall   : {rec:.4f}\n")

### Data Injection - Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, len(inject_rates) + 1, figsize=(5 * (len(inject_rates) + 1), 4))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred,
    display_labels=["Rejected", "Approved"], cmap="Blues", ax=axes[0])
axes[0].set_title(f"Baseline\nAcc={test_acc:.3f}  F1={test_f1:.3f}")

for i, rate in enumerate(inject_rates):
    cm = inject_results[rate]["cm"]
    ConfusionMatrixDisplay(cm, display_labels=["Rejected", "Approved"]).plot(
        cmap="Oranges", ax=axes[i + 1])
    axes[i + 1].set_title(
        f"Inject {rate*100:.0f}%\n"
        f"Acc={inject_results[rate]['accuracy']:.3f}  F1={inject_results[rate]['f1']:.3f}")

plt.suptitle("Data Injection - Confusion Matrices", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Attack Comparison Summary
Comparing both attacks at all poisoning levels against the baseline.

In [ ]:
rows = []
rows.append({"Attack": "Baseline", "Rate": "0%",
    "Accuracy": test_acc, "F1": test_f1, "Precision": test_prec, "Recall": test_rec})

for rate in flip_rates:
    r = flip_results[rate]
    rows.append({"Attack": "Label Flip", "Rate": f"{rate*100:.0f}%",
        "Accuracy": r["accuracy"], "F1": r["f1"], "Precision": r["precision"], "Recall": r["recall"]})

for rate in inject_rates:
    r = inject_results[rate]
    rows.append({"Attack": "Data Injection", "Rate": f"{rate*100:.0f}%",
        "Accuracy": r["accuracy"], "F1": r["f1"], "Precision": r["precision"], "Recall": r["recall"]})

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

# bar charts
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
labels = summary_df.apply(lambda r: f"{r['Attack']}\n{r['Rate']}", axis=1)
colors = ["steelblue"] + ["#e84d4d"] * len(flip_rates) + ["#f39c12"] * len(inject_rates)

for metric, ax in zip(["Accuracy", "F1"], axes):
    ax.bar(range(len(labels)), summary_df[metric], color=colors, edgecolor="black")
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel(metric)
    ax.set_title(f"{metric}: Baseline vs Attacks")
    ax.set_ylim(0, 1)
    for i, v in enumerate(summary_df[metric]):
        ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()

# degradation curve
fig, ax = plt.subplots(figsize=(10, 5))
pcts = [0, 5, 10, 20]

flip_f1s = [test_f1] + [flip_results[r]["f1"] for r in flip_rates]
flip_acc = [test_acc] + [flip_results[r]["accuracy"] for r in flip_rates]
inj_f1s  = [test_f1] + [inject_results[r]["f1"] for r in inject_rates]
inj_acc  = [test_acc] + [inject_results[r]["accuracy"] for r in inject_rates]

ax.plot(pcts, flip_f1s, "o-", color="#e84d4d", linewidth=2, label="Label Flip - F1")
ax.plot(pcts, flip_acc, "s--", color="#e84d4d", linewidth=1.5, label="Label Flip - Acc", alpha=0.7)
ax.plot(pcts, inj_f1s, "o-", color="#f39c12", linewidth=2, label="Injection - F1")
ax.plot(pcts, inj_acc, "s--", color="#f39c12", linewidth=1.5, label="Injection - Acc", alpha=0.7)

ax.set_xlabel("Poisoning Rate (%)")
ax.set_ylabel("Score")
ax.set_title("Model Degradation vs Poisoning Rate")
ax.set_xticks(pcts)
ax.set_xticklabels([f"{p}%" for p in pcts])
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## Defense with Isolation Forest
We train an Isolation Forest on the poisoned training data to detect anomalies. After removing detected outliers we retrain the Decision Tree and measure how much performance is restored.

### Defense on Label Flipping

In [ ]:
def defend_with_isolation_forest(X_tr, y_tr, contamination="auto", seed=42):
    """Train IF on training data, remove detected outliers, return cleaned data."""
    iso = IsolationForest(n_estimators=200, contamination=contamination,
                          random_state=seed, n_jobs=-1)
    preds = iso.fit_predict(X_tr)
    mask = preds == 1
    return X_tr[mask].copy(), y_tr[mask].copy(), mask, iso

defense_flip_results = {}

for rate in flip_rates:
    y_poisoned, flipped_idx = label_flipping_attack(X_train, y_train, rate)

    poison_mask = pd.Series(False, index=X_train.index)
    poison_mask.loc[flipped_idx] = True

    X_clean, y_clean, inlier_mask, iso_model = defend_with_isolation_forest(
        X_train, y_poisoned, contamination=rate)

    detected_outliers = ~inlier_mask
    n_removed        = detected_outliers.sum()
    n_poison_removed = (detected_outliers & poison_mask.values).sum()
    n_clean_removed  = (detected_outliers & ~poison_mask.values).sum()
    n_poison_total   = poison_mask.sum()
    detection_rate   = n_poison_removed / n_poison_total if n_poison_total > 0 else 0
    false_positive   = n_clean_removed / (~poison_mask).sum()

    dt_def = DecisionTreeClassifier(random_state=42, class_weight="balanced")
    dt_def.fit(X_clean, y_clean)
    y_pred_def = dt_def.predict(X_test)

    acc  = accuracy_score(y_test, y_pred_def)
    f1   = f1_score(y_test, y_pred_def)
    prec = precision_score(y_test, y_pred_def)
    rec  = recall_score(y_test, y_pred_def)

    defense_flip_results[rate] = {
        "n_removed": int(n_removed),
        "n_poison_removed": int(n_poison_removed),
        "n_clean_removed": int(n_clean_removed),
        "detection_rate": round(detection_rate, 4),
        "false_positive_rate": round(false_positive, 4),
        "train_size_after": len(y_clean),
        "accuracy": round(acc, 4), "f1": round(f1, 4),
        "precision": round(prec, 4), "recall": round(rec, 4),
        "cm": confusion_matrix(y_test, y_pred_def),
        "feature_importances": pd.Series(
            dt_def.feature_importances_, index=X_train.columns).sort_values(ascending=False),
    }

    rp = flip_results[rate]
    print(f"Flip {rate*100:.0f}% - IF Defense:")
    print(f"  Removed {n_removed} samples ({n_poison_removed} poisoned, {n_clean_removed} clean)")
    print(f"  Detection rate: {detection_rate:.2%}  |  FP rate: {false_positive:.2%}")
    print(f"  Baseline  -> Acc={test_acc:.4f}, F1={test_f1:.4f}")
    print(f"  Poisoned  -> Acc={rp['accuracy']:.4f}, F1={rp['f1']:.4f}")
    print(f"  Defended  -> Acc={acc:.4f}, F1={f1:.4f}")
    f1_restored = (f1 - rp['f1']) / (test_f1 - rp['f1']) * 100 if test_f1 != rp['f1'] else 0
    print(f"  F1 recovery: {f1_restored:.1f}%\n")

### Defense on Data Injection

In [ ]:
defense_inject_results = {}

for rate in inject_rates:
    X_inj, y_inj, n_injected = data_injection_attack(X_train, y_train, rate)

    poison_mask_inj = np.zeros(len(y_inj), dtype=bool)
    poison_mask_inj[len(y_train):] = True

    contamination_est = n_injected / len(y_inj)
    X_clean_inj, y_clean_inj, inlier_mask_inj, _ = defend_with_isolation_forest(
        X_inj, y_inj, contamination=contamination_est)

    detected_outliers = ~inlier_mask_inj
    n_removed        = detected_outliers.sum()
    n_poison_removed = (detected_outliers & poison_mask_inj).sum()
    n_clean_removed  = (detected_outliers & ~poison_mask_inj).sum()
    n_poison_total   = poison_mask_inj.sum()
    detection_rate   = n_poison_removed / n_poison_total if n_poison_total > 0 else 0
    false_positive   = n_clean_removed / (~poison_mask_inj).sum()

    dt_def_inj = DecisionTreeClassifier(random_state=42, class_weight="balanced")
    dt_def_inj.fit(X_clean_inj, y_clean_inj)
    y_pred_def_inj = dt_def_inj.predict(X_test)

    acc  = accuracy_score(y_test, y_pred_def_inj)
    f1   = f1_score(y_test, y_pred_def_inj)
    prec = precision_score(y_test, y_pred_def_inj)
    rec  = recall_score(y_test, y_pred_def_inj)

    defense_inject_results[rate] = {
        "n_removed": int(n_removed),
        "n_poison_removed": int(n_poison_removed),
        "n_clean_removed": int(n_clean_removed),
        "detection_rate": round(detection_rate, 4),
        "false_positive_rate": round(false_positive, 4),
        "train_size_after": len(y_clean_inj),
        "accuracy": round(acc, 4), "f1": round(f1, 4),
        "precision": round(prec, 4), "recall": round(rec, 4),
        "cm": confusion_matrix(y_test, y_pred_def_inj),
        "feature_importances": pd.Series(
            dt_def_inj.feature_importances_, index=X_train.columns).sort_values(ascending=False),
    }

    rp = inject_results[rate]
    print(f"Inject {rate*100:.0f}% - IF Defense:")
    print(f"  Removed {n_removed} samples ({n_poison_removed} poisoned, {n_clean_removed} clean)")
    print(f"  Detection rate: {detection_rate:.2%}  |  FP rate: {false_positive:.2%}")
    print(f"  Baseline  -> Acc={test_acc:.4f}, F1={test_f1:.4f}")
    print(f"  Poisoned  -> Acc={rp['accuracy']:.4f}, F1={rp['f1']:.4f}")
    print(f"  Defended  -> Acc={acc:.4f}, F1={f1:.4f}\n")

### Detection Rate Analysis
How well does Isolation Forest actually find the poisoned samples?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# label flipping detection
rates_pct = [r * 100 for r in flip_rates]
det_rates = [defense_flip_results[r]["detection_rate"] * 100 for r in flip_rates]
fp_rates  = [defense_flip_results[r]["false_positive_rate"] * 100 for r in flip_rates]

x = np.arange(len(flip_rates))
w = 0.3
axes[0].bar(x - w/2, det_rates, w, color="#2ecc71", edgecolor="black", label="Detection Rate")
axes[0].bar(x + w/2, fp_rates,  w, color="#e84d4d", edgecolor="black", label="False Positive Rate")
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"{r:.0f}%" for r in rates_pct])
axes[0].set_xlabel("Flip Rate")
axes[0].set_ylabel("Rate (%)")
axes[0].set_title("IF Detection - Label Flipping")
axes[0].legend()
axes[0].set_ylim(0, 100)
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f"{bar.get_height():.1f}%", ha="center", fontsize=9)

# data injection detection
det_rates_inj = [defense_inject_results[r]["detection_rate"] * 100 for r in inject_rates]
fp_rates_inj  = [defense_inject_results[r]["false_positive_rate"] * 100 for r in inject_rates]

axes[1].bar(x - w/2, det_rates_inj, w, color="#2ecc71", edgecolor="black", label="Detection Rate")
axes[1].bar(x + w/2, fp_rates_inj,  w, color="#e84d4d", edgecolor="black", label="False Positive Rate")
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"{r*100:.0f}%" for r in inject_rates])
axes[1].set_xlabel("Injection Rate")
axes[1].set_ylabel("Rate (%)")
axes[1].set_title("IF Detection - Data Injection")
axes[1].legend()
axes[1].set_ylim(0, 100)
for bar in axes[1].patches:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f"{bar.get_height():.1f}%", ha="center", fontsize=9)

plt.suptitle("Isolation Forest Detection Rates", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Feature Importance Comparison
Comparing how feature importances shift between the baseline, poisoned (20% flip), and defended models.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
feature_names = X_train.columns.tolist()

baseline_imp = importances
rate_show = 0.20

# poisoned model at 20% flip
y_poisoned_20, _ = label_flipping_attack(X_train, y_train, rate_show)
dt_poisoned_20 = DecisionTreeClassifier(random_state=42, class_weight="balanced")
dt_poisoned_20.fit(X_train, y_poisoned_20)
poisoned_imp = pd.Series(dt_poisoned_20.feature_importances_, index=feature_names).sort_values(ascending=False)

defended_imp = defense_flip_results[rate_show]["feature_importances"]
order = baseline_imp.index

colors_fi = ["steelblue", "#e84d4d", "#2ecc71"]
titles_fi = ["Baseline", "Poisoned (Flip 20%)", "Defended (IF)"]
imps = [baseline_imp, poisoned_imp.reindex(order), defended_imp.reindex(order)]

for ax, imp, color, title in zip(axes, imps, colors_fi, titles_fi):
    ax.barh(range(len(order)), imp.values, color=color, edgecolor="black", alpha=0.85)
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels(order, fontsize=10)
    ax.invert_yaxis()
    ax.set_xlabel("Importance")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlim(0, max(baseline_imp.max(), poisoned_imp.max(), defended_imp.max()) * 1.15)

plt.suptitle("Feature Importance: Baseline vs Poisoned vs Defended", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# numerical table
fi_comparison = pd.DataFrame({
    "Baseline": baseline_imp.reindex(order).round(4),
    "Poisoned 20%": poisoned_imp.reindex(order).round(4),
    "Defended 20%": defended_imp.reindex(order).round(4),
})
fi_comparison["Delta Poison"] = (fi_comparison.iloc[:, 1] - fi_comparison.iloc[:, 0]).round(4)
fi_comparison["Delta Defended"] = (fi_comparison.iloc[:, 2] - fi_comparison.iloc[:, 0]).round(4)
print(fi_comparison.to_string())

### Full Comparison: Clean vs Poisoned vs Defended
Summary table and visualizations comparing all scenarios.

In [ ]:
all_rows = []
all_rows.append({"Scenario": "Baseline", "Rate": "0%",
                 "Accuracy": test_acc, "F1": test_f1,
                 "Precision": test_prec, "Recall": test_rec})

for rate in flip_rates:
    rp = flip_results[rate]
    rd = defense_flip_results[rate]
    all_rows.append({"Scenario": f"Flip {rate*100:.0f}% Poisoned", "Rate": f"{rate*100:.0f}%",
        "Accuracy": rp["accuracy"], "F1": rp["f1"], "Precision": rp["precision"], "Recall": rp["recall"]})
    all_rows.append({"Scenario": f"Flip {rate*100:.0f}% Defended", "Rate": f"{rate*100:.0f}%",
        "Accuracy": rd["accuracy"], "F1": rd["f1"], "Precision": rd["precision"], "Recall": rd["recall"]})

for rate in inject_rates:
    rp = inject_results[rate]
    rd = defense_inject_results[rate]
    all_rows.append({"Scenario": f"Inject {rate*100:.0f}% Poisoned", "Rate": f"{rate*100:.0f}%",
        "Accuracy": rp["accuracy"], "F1": rp["f1"], "Precision": rp["precision"], "Recall": rp["recall"]})
    all_rows.append({"Scenario": f"Inject {rate*100:.0f}% Defended", "Rate": f"{rate*100:.0f}%",
        "Accuracy": rd["accuracy"], "F1": rd["f1"], "Precision": rd["precision"], "Recall": rd["recall"]})

full_df = pd.DataFrame(all_rows)
print(full_df.to_string(index=False))

# grouped bar charts
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

scenarios_flip = ["Baseline"] + [f"Flip {r*100:.0f}%" for r in flip_rates]
poisoned_vals  = [test_f1] + [flip_results[r]["f1"] for r in flip_rates]
defended_vals  = [test_f1] + [defense_flip_results[r]["f1"] for r in flip_rates]

x = np.arange(len(scenarios_flip))
w = 0.25
axes[0].bar(x - w/2, poisoned_vals, w, color="#e84d4d", edgecolor="black", label="Poisoned")
axes[0].bar(x + w/2, defended_vals, w, color="#2ecc71", edgecolor="black", label="Defended")
axes[0].axhline(y=test_f1, color="steelblue", linestyle="--", alpha=0.5, label="Baseline")
axes[0].set_xticks(x)
axes[0].set_xticklabels(scenarios_flip)
axes[0].set_ylabel("F1 Score")
axes[0].set_title("Label Flipping: Poisoned vs Defended")
axes[0].legend()
axes[0].set_ylim(0, 1)

scenarios_inj = ["Baseline"] + [f"Inject {r*100:.0f}%" for r in inject_rates]
poisoned_vals_inj = [test_f1] + [inject_results[r]["f1"] for r in inject_rates]
defended_vals_inj = [test_f1] + [defense_inject_results[r]["f1"] for r in inject_rates]

x = np.arange(len(scenarios_inj))
axes[1].bar(x - w/2, poisoned_vals_inj, w, color="#f39c12", edgecolor="black", label="Poisoned")
axes[1].bar(x + w/2, defended_vals_inj, w, color="#2ecc71", edgecolor="black", label="Defended")
axes[1].axhline(y=test_f1, color="steelblue", linestyle="--", alpha=0.5, label="Baseline")
axes[1].set_xticks(x)
axes[1].set_xticklabels(scenarios_inj)
axes[1].set_ylabel("F1 Score")
axes[1].set_title("Data Injection: Poisoned vs Defended")
axes[1].legend()
axes[1].set_ylim(0, 1)

plt.suptitle("Defense Effectiveness", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# confusion matrices for worst case (20% flip)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred,
    display_labels=["Rejected", "Approved"], cmap="Blues", ax=axes[0])
axes[0].set_title(f"Baseline\nF1={test_f1:.3f}")

cm_p = flip_results[0.20]["cm"]
ConfusionMatrixDisplay(cm_p, display_labels=["Rejected", "Approved"]).plot(cmap="Reds", ax=axes[1])
axes[1].set_title(f"Poisoned (Flip 20%)\nF1={flip_results[0.20]['f1']:.3f}")

cm_d = defense_flip_results[0.20]["cm"]
ConfusionMatrixDisplay(cm_d, display_labels=["Rejected", "Approved"]).plot(cmap="Greens", ax=axes[2])
axes[2].set_title(f"Defended (IF)\nF1={defense_flip_results[0.20]['f1']:.3f}")

plt.suptitle("Worst Case (20% Flip): Baseline vs Poisoned vs Defended", fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

## Explainability with LIME and ELI5
We use LIME and ELI5 to understand how the model makes decisions and how poisoning changes the decision logic. We compare explanations from the baseline, poisoned, and defended models.

### ELI5 - Global Feature Weights
ELI5 shows the global feature weights learned by each model. We compare the baseline, poisoned (20% flip), and defended models side by side.

In [ ]:
# baseline model weights
print("=== Baseline Model ===")
eli5_baseline = eli5.explain_weights(dt, feature_names=X_train.columns.tolist())
print(eli5.format_as_text(eli5_baseline))

# poisoned model (20% flip)
y_p20, _ = label_flipping_attack(X_train, y_train, 0.20)
dt_p20 = DecisionTreeClassifier(random_state=42, class_weight="balanced")
dt_p20.fit(X_train, y_p20)

print("\n=== Poisoned Model (20% Flip) ===")
eli5_poisoned = eli5.explain_weights(dt_p20, feature_names=X_train.columns.tolist())
print(eli5.format_as_text(eli5_poisoned))

# defended model
X_cl, y_cl, _, _ = defend_with_isolation_forest(X_train, y_p20, contamination=0.20)
dt_d20 = DecisionTreeClassifier(random_state=42, class_weight="balanced")
dt_d20.fit(X_cl, y_cl)

print("\n=== Defended Model (IF + 20% Flip) ===")
eli5_defended = eli5.explain_weights(dt_d20, feature_names=X_train.columns.tolist())
print(eli5.format_as_text(eli5_defended))

### LIME - Individual Prediction Explanations
LIME explains individual predictions by approximating the model locally with a simple interpretable model. We pick a few test samples and compare how the baseline vs poisoned model explains the same prediction.

In [ ]:
explainer = LimeTabularExplainer(
    X_train.values,
    feature_names=X_train.columns.tolist(),
    class_names=["Rejected", "Approved"],
    mode="classification",
    random_state=42,
)

# pick 3 test samples: one approved, one rejected, one where poisoned model disagrees
approved_idx = np.where(y_pred == 1)[0][0]
rejected_idx = np.where(y_pred == 0)[0][0]

y_pred_p20 = dt_p20.predict(X_test)
disagree_mask = y_pred != y_pred_p20
disagree_idx = np.where(disagree_mask)[0][0] if disagree_mask.any() else approved_idx

sample_indices = [rejected_idx, approved_idx, disagree_idx]
sample_labels = ["Rejected sample", "Approved sample", "Disagreement sample"]

fig, axes = plt.subplots(len(sample_indices), 2, figsize=(16, 5 * len(sample_indices)))

for row, (idx, label) in enumerate(zip(sample_indices, sample_labels)):
    instance = X_test.iloc[idx].values

    # baseline explanation
    exp_base = explainer.explain_instance(instance, dt.predict_proba, num_features=8)
    base_features = exp_base.as_list()

    # poisoned explanation
    exp_poison = explainer.explain_instance(instance, dt_p20.predict_proba, num_features=8)
    poison_features = exp_poison.as_list()

    # plot baseline
    feat_names_b = [f[0] for f in base_features]
    weights_b = [f[1] for f in base_features]
    colors_b = ["#2ecc71" if w > 0 else "#e84d4d" for w in weights_b]
    axes[row, 0].barh(feat_names_b, weights_b, color=colors_b, edgecolor="black")
    axes[row, 0].set_title(f"{label} - Baseline\n(pred={dt.predict([instance])[0]})")
    axes[row, 0].set_xlabel("Weight")

    # plot poisoned
    feat_names_p = [f[0] for f in poison_features]
    weights_p = [f[1] for f in poison_features]
    colors_p = ["#2ecc71" if w > 0 else "#e84d4d" for w in weights_p]
    axes[row, 1].barh(feat_names_p, weights_p, color=colors_p, edgecolor="black")
    axes[row, 1].set_title(f"{label} - Poisoned (20% Flip)\n(pred={dt_p20.predict([instance])[0]})")
    axes[row, 1].set_xlabel("Weight")

plt.suptitle("LIME: Baseline vs Poisoned Explanations", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### LIME - Defended Model Comparison
We compare the same sample's explanation across all three models: baseline, poisoned, and defended.

In [ ]:
# pick the disagreement sample for a detailed 3-way comparison
instance = X_test.iloc[disagree_idx].values
models = [dt, dt_p20, dt_d20]
model_names = ["Baseline", "Poisoned (20% Flip)", "Defended (IF)"]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, model, name in zip(axes, models, model_names):
    exp = explainer.explain_instance(instance, model.predict_proba, num_features=8)
    features = exp.as_list()
    feat_names = [f[0] for f in features]
    weights = [f[1] for f in features]
    colors = ["#2ecc71" if w > 0 else "#e84d4d" for w in weights]

    ax.barh(feat_names, weights, color=colors, edgecolor="black")
    pred = model.predict([instance])[0]
    prob = model.predict_proba([instance])[0]
    ax.set_title(f"{name}\npred={pred} (prob={prob[1]:.2f})")
    ax.set_xlabel("Weight")

plt.suptitle("LIME: Baseline vs Poisoned vs Defended (same sample)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# print actual vs predicted for context
print(f"Sample index: {disagree_idx}")
print(f"True label: {y_test.iloc[disagree_idx]}")
print(f"Baseline prediction: {dt.predict([instance])[0]}")
print(f"Poisoned prediction: {dt_p20.predict([instance])[0]}")
print(f"Defended prediction: {dt_d20.predict([instance])[0]}")